# TFM · Extracción exploratoria de backlogs de Jira (notebook de apoyo)
Herramienta auxiliar del framework de planificación de sprints. Extrae, normaliza y valida backlogs de **instancias públicas de Jira de proyectos de código abierto** (y también de Jira Cloud), para explorar backlogs reales distintos de los escenarios E1–E3. No forma parte del flujo principal: la extracción de los escenarios experimentales se hace con `TFM_Fase3_extraccion.ipynb` y el módulo `jira_ingesta.py`.

Funciona con dos tipos de Jira:
- **Cloud** (`*.atlassian.net`, tu sitio): API v3, con correo y token.
- **Server / Data Center públicos** (Apache, MongoDB, Qt…): API v2, acceso anónimo.

**Con Jira Cloud:** en Colab, abre *Secrets*, crea `JIRA_URL`, `JIRA_EMAIL` y `JIRA_API_TOKEN`, y activa *Notebook access*. Los sitios públicos no necesitan credenciales.

## 1. Librerías

In [ ]:
import json, math, os, re, time
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_colwidth", 60)
os.makedirs("salidas", exist_ok=True)
print("requests", requests.__version__, "| pandas", pd.__version__)

## 2. Sitios y configuración
- `SITIOS`: los Jira disponibles. Los públicos pueden cambiar o cerrar el acceso anónimo; la celda 4 te dice si responden.
- `SITIO` y `PROYECTO`: el backlog de la prueba actual.
- `CONFIG`: cómo leer ese backlog. La celda 4 te ayuda a llenarla.

In [ ]:
SITIOS = {
    "mi_cloud": {"url": None, "tipo": "cloud"},   # se completa con tus Secrets
    "apache":   {"url": "https://issues.apache.org/jira", "tipo": "server"},
    "mongodb":  {"url": "https://jira.mongodb.org",       "tipo": "server"},
    "qt":       {"url": "https://bugreports.qt.io",       "tipo": "server"},
}

# ---------- Prueba actual ----------
SITIO     = "apache"     # clave de SITIOS
PROYECTO  = "MESOS"      # clave del proyecto en ese sitio
VELOCIDAD = 25           # story points por sprint
MAX_ISSUES = 2000        # tope de issues a descargar (los backlogs públicos son enormes)

CONFIG = {
    # Nombres o IDs ("customfield_10016") a probar, en orden
    "story_points":  ["Story point estimate", "Story Points", "Story Point",
                      "Estimación de puntos de historia", "Puntos de historia"],
    "valor_negocio": ["Valor de negocio", "Business Value"],
    # Tipos de vínculo de dependencia. None = autodetectar ("blocks", "depends on"...).
    # Manual: {"Blocks": "inward", "Dependent": "outward"} (lado donde está el prerrequisito)
    "vinculos_dependencia": None,
    # Excluye épicas y subtareas
    "solo_nivel_historia": True,
    # Solo estos tipos de issue (lista vacía = todos), p. ej. ["Story", "Improvement"]
    "tipos": [],
    # Qué significa "pendiente". None = automático (Cloud: statusCategory; Server: resolution)
    "jql_pendientes": None,
    # Filtro JQL adicional, p. ej. 'AND "Story Points" is not EMPTY'
    "jql_extra": "",
    # Si falta el valor de negocio, estimarlo desde la prioridad (prioridad_num * 20)
    "valor_desde_prioridad": False,
}


def leer_config(nombre, secreto=False):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:
        pass
    return getpass(f"{nombre}: ") if secreto else input(f"{nombre}: ")


CREDENCIALES = {}
def credenciales_cloud():
    if not CREDENCIALES:
        CREDENCIALES["url"] = leer_config("JIRA_URL").rstrip("/")
        CREDENCIALES["email"] = leer_config("JIRA_EMAIL")
        CREDENCIALES["token"] = leer_config("JIRA_API_TOKEN", secreto=True)
        SITIOS["mi_cloud"]["url"] = CREDENCIALES["url"]
    return CREDENCIALES

print("Prueba:", SITIO, "/", PROYECTO, "| Velocidad:", VELOCIDAD, "| Máx. issues:", MAX_ISSUES)

## 3. Cliente de Jira (Cloud v3 y Server v2)

In [ ]:
PRIORIDAD_NUM = {"Highest": 5, "High": 4, "Medium": 3, "Low": 2, "Lowest": 1,
                 "Blocker": 5, "Critical": 4, "Major": 3, "Minor": 2, "Trivial": 1,
                 "Más alta": 5, "Alta": 4, "Media": 3, "Baja": 2, "Más baja": 1}
# Textos de vínculo que indican "X depende de Y"
PATRONES_DEP = ("is blocked by", "blocked by", "depends on", "depends upon", "bloqueada por",
                "bloqueado por", "depende de")


class Jira:
    def __init__(self, url, tipo="cloud", email=None, token=None, pausa=0.3):
        self.base = url.rstrip("/")
        self.tipo = tipo
        self.api = "/rest/api/3" if tipo == "cloud" else "/rest/api/2"
        self.pausa = pausa
        self.s = requests.Session()
        self.s.headers.update({"Accept": "application/json", "Content-Type": "application/json"})
        if email and token:
            self.s.auth = (email, token)                       # Cloud
        elif token:
            self.s.headers["Authorization"] = f"Bearer {token}"  # Server/DC con token personal
        self._campos = None

    def _req(self, metodo, ruta, **kw):
        for intento in range(5):
            r = self.s.request(metodo, f"{self.base}{self.api}{ruta}", timeout=60, **kw)
            if r.status_code in (429, 503):                    # límite de peticiones: esperar y reintentar
                espera = int(r.headers.get("Retry-After", 5 * (intento + 1)))
                print(f"  Límite de peticiones, esperando {espera}s...")
                time.sleep(espera)
                continue
            if r.status_code >= 400:
                raise RuntimeError(f"{metodo} {ruta} -> {r.status_code}: {r.text[:500]}")
            return r.json() if r.content else None
        raise RuntimeError(f"{metodo} {ruta}: demasiados reintentos")

    def get(self, ruta, **kw):
        return self._req("GET", ruta, **kw)

    def post(self, ruta, json):
        return self._req("POST", ruta, json=json)

    def proyectos(self):
        if self.tipo == "cloud":
            return self.get("/project/search", params={"maxResults": 100})["values"]
        return self.get("/project")

    def tipos_vinculo(self):
        return self.get("/issueLinkType")["issueLinkTypes"]

    def todos_los_campos(self):
        if self._campos is None:
            self._campos = self.get("/field")
        return self._campos

    def resolver_campo(self, candidatos):
        campos = self.todos_los_campos()
        ids = {f["id"] for f in campos}
        for c in candidatos:
            if c in ids:
                return c
            coincidencias = [f["id"] for f in campos if f["name"] == c]
            if len(coincidencias) > 1:
                print(f"⚠️ Hay {len(coincidencias)} campos llamados '{c}': {coincidencias}. "
                      f"Se usa {coincidencias[0]}; si no es el correcto, pon su ID en CONFIG.")
            if coincidencias:
                return coincidencias[0]
        return None

    def campos(self, config):
        return {"story_points": self.resolver_campo(config["story_points"]),
                "valor_negocio": self.resolver_campo(config["valor_negocio"]),
                "sprint": self.resolver_campo(["Sprint"])}

    def vinculos(self, config):
        """{nombre_tipo: 'inward'|'outward'}: lado del vínculo donde está el prerrequisito."""
        if config["vinculos_dependencia"]:
            return dict(config["vinculos_dependencia"])
        mapa = {}
        for t in self.tipos_vinculo():
            if any(p in t["inward"].lower() for p in PATRONES_DEP):
                mapa[t["name"]] = "inward"
            elif any(p in t["outward"].lower() for p in PATRONES_DEP):
                mapa[t["name"]] = "outward"
        return mapa

    def buscar(self, jql, campos, maximo):
        issues = []
        if self.tipo == "cloud":
            token = None
            while len(issues) < maximo:
                cuerpo = {"jql": jql, "fields": campos, "maxResults": 100}
                if token:
                    cuerpo["nextPageToken"] = token
                r = self.post("/search/jql", cuerpo)
                issues += r.get("issues", [])
                token = r.get("nextPageToken")
                if r.get("isLast", True) or not token:
                    break
        else:
            inicio = 0
            while len(issues) < maximo:
                r = self.post("/search", {"jql": jql, "fields": campos,
                                          "startAt": inicio, "maxResults": 100})
                lote = r.get("issues", [])
                issues += lote
                inicio += len(lote)
                if r.get("total") is not None and len(issues) == len(lote):
                    print(f"  Total en Jira: {r['total']} issues")
                if not lote or inicio >= r.get("total", 0):
                    break
                time.sleep(self.pausa)                         # ser amable con los servidores públicos
        return issues[:maximo]


_clientes = {}
def cliente(sitio):
    if sitio not in _clientes:
        s = SITIOS[sitio]
        if s["tipo"] == "cloud":
            c = credenciales_cloud()
            _clientes[sitio] = Jira(c["url"], "cloud", c["email"], c["token"])
        else:
            _clientes[sitio] = Jira(s["url"], "server", token=s.get("token"))
    return _clientes[sitio]

## 4. Explorar un sitio
Comprueba que el sitio responde y muestra sus proyectos, tipos de vínculo y campos candidatos. Úsalo para elegir `PROYECTO` y ajustar `CONFIG`.

In [ ]:
FILTRO_PROYECTOS = ""   # texto para buscar proyectos por nombre o clave, p. ej. "mesos"

jira = cliente(SITIO)
proyectos = jira.proyectos()
tabla = pd.DataFrame([{"clave": p["key"], "nombre": p["name"]} for p in proyectos])
if FILTRO_PROYECTOS:
    tabla = tabla[tabla.apply(lambda r: FILTRO_PROYECTOS.lower() in (r.clave + r.nombre).lower(), axis=1)]
print(f"{SITIO}: {len(proyectos)} proyectos accesibles (mostrando {min(len(tabla), 30)})")
display(tabla.head(30))

print("Tipos de vínculo (✔ = se usará como dependencia):")
dep = jira.vinculos(CONFIG)
display(pd.DataFrame([{"nombre": t["name"], "salida": t["outward"], "entrada": t["inward"],
                       "dependencia": f"✔ ({dep[t['name']]})" if t["name"] in dep else ""}
                      for t in jira.tipos_vinculo()]))

palabras = ("point", "punto", "valor", "value", "estim", "sprint")
print("Campos candidatos:")
display(pd.DataFrame([{"id": f["id"], "nombre": f["name"], "tipo": f.get("schema", {}).get("type")}
                      for f in jira.todos_los_campos()
                      if any(p in f["name"].lower() for p in palabras)]))
print("Campos que se usarán:", jira.campos(CONFIG))

## 5. Funciones de extracción, normalización y validación

In [ ]:
def jql_backlog(jira, proyecto, config):
    pendientes = config["jql_pendientes"] or (
        "statusCategory != Done" if jira.tipo == "cloud" else "resolution = Unresolved")
    tipos = ""
    if config["tipos"]:
        tipos = "AND issuetype in (" + ", ".join(f'"{t}"' for t in config["tipos"]) + ")"
    return f'project = "{proyecto}" AND {pendientes} {tipos} {config["jql_extra"]} ORDER BY key ASC'


def extraer(jira, proyecto, campos, config, maximo):
    lista = ["summary", "issuetype", "status", "priority", "labels", "issuelinks", "created"]
    lista += [c for c in campos.values() if c]
    jql = jql_backlog(jira, proyecto, config)
    print("  JQL:", jql)
    return jira.buscar(jql, lista, maximo)


def _terminada(issue_vinculada):
    cat = issue_vinculada.get("fields", {}).get("status", {}).get("statusCategory", {})
    return cat.get("key") == "done"


def _nombre_sprint(s):
    if isinstance(s, dict):
        return s.get("name")
    m = re.search(r"name=([^,\]]+)", str(s))      # formato antiguo de Jira Server
    return m.group(1) if m else str(s)


def _es_nivel_historia(tipo):
    if tipo.get("subtask"):
        return False
    if "hierarchyLevel" in tipo:
        return tipo["hierarchyLevel"] == 0
    return tipo.get("name", "").lower() not in ("epic", "épica")


def normalizar(issues, campos, vinculos, config):
    filas = []
    for it in issues:
        f = it["fields"]
        tipo = f["issuetype"]
        if config["solo_nivel_historia"] and not _es_nivel_historia(tipo):
            continue
        prioridad = (f.get("priority") or {}).get("name", "")
        prioridad_num = PRIORIDAD_NUM.get(prioridad)
        sprints = f.get(campos.get("sprint") or "", None) or []
        pendientes, resueltas = [], []
        for l in f.get("issuelinks", []):
            lado = vinculos.get(l["type"]["name"])
            clave_lado = f"{lado}Issue" if lado else None
            if clave_lado and clave_lado in l:
                y = l[clave_lado]
                (resueltas if _terminada(y) else pendientes).append(y["key"])
        valor = f.get(campos.get("valor_negocio") or "")
        origen = "jira" if valor not in (None, "") else None
        if origen is None and config["valor_desde_prioridad"] and prioridad_num:
            valor, origen = prioridad_num * 20, "prioridad"
        filas.append({
            "clave": it["key"],
            "resumen": f["summary"],
            "tipo": tipo["name"],
            "estado": f["status"]["name"],
            "prioridad": prioridad,
            "prioridad_num": prioridad_num,
            "story_points": f.get(campos.get("story_points") or ""),
            "valor_negocio": valor,
            "valor_origen": origen,
            "etiquetas": f.get("labels", []),
            "depende_de": sorted(set(pendientes)),
            "deps_resueltas": sorted(set(resueltas)),
            "sprint": _nombre_sprint(sprints[-1]) if sprints else None,
            "creado": (f.get("created") or "")[:10],
        })
    return filas


def validar(filas):
    avisos = []          # (categoría, detalle)
    claves = {r["clave"] for r in filas}
    for r in filas:
        if r["story_points"] in (None, ""):
            avisos.append(("sin story points", r["clave"]))
        if r["valor_negocio"] in (None, ""):
            avisos.append(("sin valor de negocio", r["clave"]))
        for d in r["depende_de"]:
            if d not in claves:
                avisos.append(("dependencia pendiente fuera del backlog", f"{r['clave']} -> {d}"))
    grafo = {r["clave"]: [d for d in r["depende_de"] if d in claves] for r in filas}
    estado = {}

    def visitar(n, camino):
        estado[n] = 1
        for m in grafo[n]:
            if estado.get(m) == 1:
                avisos.append(("ciclo de dependencias", " -> ".join(camino + [n, m])))
            elif m not in estado:
                visitar(m, camino + [n])
        estado[n] = 2

    for n in grafo:
        if n not in estado:
            visitar(n, [])
    return avisos


def mostrar_avisos(avisos, ejemplos=5):
    if not avisos:
        print("Validación: sin problemas ✅")
        return
    print(f"Validación: {len(avisos)} aviso(s)")
    por_cat = {}
    for cat, det in avisos:
        por_cat.setdefault(cat, []).append(det)
    for cat, dets in por_cat.items():
        extra = f" … (+{len(dets) - ejemplos})" if len(dets) > ejemplos else ""
        print(f"  - {cat}: {len(dets)} → {', '.join(dets[:ejemplos])}{extra}")


def resumen(etiqueta, df, avisos, velocidad):
    sp = pd.to_numeric(df["story_points"], errors="coerce")
    vn = pd.to_numeric(df["valor_negocio"], errors="coerce")
    total = sp.fillna(0).sum()
    cats = [c for c, _ in avisos]
    return {"backlog": etiqueta, "historias": len(df), "story_points": total,
            "pct_con_sp": round(100 * sp.notna().mean(), 1),
            "pct_con_valor": round(100 * vn.notna().mean(), 1),
            "con_dependencias": int((df["depende_de"].str.len() > 0).sum()),
            "deps_externas": cats.count("dependencia pendiente fuera del backlog"),
            "ciclos": cats.count("ciclo de dependencias"),
            "sprints_min": math.ceil(total / velocidad) if total else 0}


def guardar(etiqueta, datos, filas):
    base = f"salidas/{etiqueta}"
    with open(f"{base}_respuesta_jira.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False)
    with open(f"{base}_backlog.json", "w", encoding="utf-8") as f:
        json.dump(filas, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame(filas)
    for col in ("etiquetas", "depende_de", "deps_resueltas"):
        df[col] = df[col].str.join(",")
    df.to_csv(f"{base}_backlog.csv", index=False, encoding="utf-8-sig")
    return base


def procesar(sitio, proyecto, config=CONFIG, velocidad=VELOCIDAD, maximo=MAX_ISSUES, datos=None):
    """Pipeline completo para un backlog. Si pasas 'datos', trabaja sin conexión."""
    etiqueta = f"{sitio}_{proyecto}"
    if datos is None:
        jira = cliente(sitio)
        campos = jira.campos(config)
        vinculos = jira.vinculos(config)
        print(f"[{etiqueta}] campos: {campos} | vínculos de dependencia: {vinculos}")
        datos = {"sitio": sitio, "proyecto": proyecto, "campos": campos, "vinculos": vinculos,
                 "issues": extraer(jira, proyecto, campos, config, maximo)}
    filas = normalizar(datos["issues"], datos["campos"], datos["vinculos"], config)
    if not filas:
        print(f"[{etiqueta}] no hay issues pendientes con los filtros actuales.")
        return None, [], None
    df = pd.DataFrame(filas)
    avisos = validar(filas)
    base = guardar(etiqueta, datos, filas)
    print(f"[{etiqueta}] {len(datos['issues'])} descargados, {len(df)} en el backlog → {base}_backlog.csv")
    return df, avisos, resumen(etiqueta, df, avisos, velocidad)

print("Funciones listas")

## 6. Extraer un backlog

In [ ]:
df, avisos, res = procesar(SITIO, PROYECTO)
df

## 7. Validación y resumen

In [ ]:
mostrar_avisos(avisos)
display(pd.DataFrame([res]))

sp = pd.to_numeric(df["story_points"], errors="coerce")
print("Por tipo de issue:")
display(df.assign(story_points=sp).groupby("tipo")["story_points"]
          .agg(issues="size", con_sp="count", story_points="sum"))
print("Por prioridad:")
display(df.assign(story_points=sp).groupby("prioridad")["story_points"]
          .agg(issues="size", con_sp="count", story_points="sum"))

## 8. Comparar varios backlogs
Cada prueba es `(sitio, proyecto, config)`. Para un proyecto con otros nombres de campos, usa `{**CONFIG, "story_points": ["customfield_XXXXX"]}`.
Para el TFM conviene quedarse con issues estimadas: `{**CONFIG, "jql_extra": 'AND "Story Points" is not EMPTY'}` (ajusta el nombre del campo).

In [ ]:
PRUEBAS = [
    ("apache",  "MESOS",  CONFIG),
    ("mongodb", "SERVER", {**CONFIG, "tipos": ["Task", "Improvement", "New Feature"]}),
    # ("mi_cloud", "TDA", CONFIG),
    # ("qt", "QTBUG", {**CONFIG, "valor_desde_prioridad": True}),
]

resultados, backlogs = [], {}
for sitio, proyecto, cfg in PRUEBAS:
    try:
        d, av, r = procesar(sitio, proyecto, config=cfg)
        if d is not None:
            backlogs[f"{sitio}_{proyecto}"] = d
            resultados.append(r)
    except Exception as e:
        print(f"[{sitio}_{proyecto}] error: {e}")

comparacion = pd.DataFrame(resultados)
comparacion.to_csv("salidas/comparacion_backlogs.csv", index=False, encoding="utf-8-sig")
comparacion

## 9. (Opcional) Descargar o guardar en Google Drive

In [ ]:
import shutil
shutil.make_archive("salidas_backlogs", "zip", "salidas")

# Opción A: descargar todo en un zip
from google.colab import files
files.download("salidas_backlogs.zip")

# Opción B: copiar a Drive (descomenta)
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree("salidas", "/content/drive/MyDrive/TFM/datos", dirs_exist_ok=True)

## 10. (Opcional) Reprocesar sin conexión
Sube `<sitio>_<proyecto>_respuesta_jira.json` a `salidas/` y ejecuta:

In [ ]:
sitio_off, proyecto_off = SITIO, PROYECTO
with open(f"salidas/{sitio_off}_{proyecto_off}_respuesta_jira.json", encoding="utf-8") as f:
    datos_off = json.load(f)
df, avisos, res = procesar(sitio_off, proyecto_off, datos=datos_off)
mostrar_avisos(avisos)
display(pd.DataFrame([res]))